<a href="https://colab.research.google.com/github/piaoruilin/hallucination-medical-vlm/blob/main/inject_hallucinations_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inject hallucinations into VQA answers with Qwen (token-level labels)
Runtime > GPU. Input: a CSV with a question column and a correct-answer column.
Output: a JSONL file with answer text, word tokens, and 0/1 hallucination labels.

In [ ]:
!pip install -q transformers accelerate bitsandbytes tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 79.6 MB/s eta 0:00:00


In [ ]:
import torch, importlib.metadata as md
print("CUDA available:", torch.cuda.is_available())
try:
    print("bitsandbytes version:", md.version("bitsandbytes"))
except Exception as e:
    print("bitsandbytes not installed:", e)
!nvidia-smi

CUDA available: True
bitsandbytes version: 0.50.2
Fri Oct 16 12:48:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             50W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled 

## 1. Load Qwen (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Check the Qwen model page for the newest instruct version your supervisor recommends.
# Use a 3B model if 7B is too slow or runs out of memory.
model_id = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
llm.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

## 2. Load your data
Edit the path and column names to match your file. For PMC-VQA the question and answer columns are probably
`Question` and `Answer`; print `df.columns` to check.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import re
import pandas as pd
from huggingface_hub import hf_hub_download

# Text-only task, so only the CSV is needed (no images)
CSV_PATH = hf_hub_download("xmcmic/PMC-VQA", "train_2.csv", repo_type="dataset")
Q_COL = "Question"

df = pd.read_csv(CSV_PATH)
print(df.columns.tolist(), len(df))

# Answer column (the README spells it "Anwser"), then swap letters for the option text
A_RAW = next(c for c in df.columns if c.lower() in ("answer", "anwser"))
choice_col = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}

def get_answer(row):
    a = str(row[A_RAW]).strip()
    if re.fullmatch(r"[A-Da-d]", a):                      # it's just a letter
        col = choice_col.get("choice" + a.lower())
        if col and pd.notna(row[col]):
            a = str(row[col]).strip()
    return re.sub(r"^[A-D][\.\:\)]\s*", "", a)            # drop a leading "B:" / "B."

df["ans"] = df.apply(get_answer, axis=1)
A_COL = "ans"        # the later cells read A_COL, so they work unchanged
print((df["ans"].str.len() <= 2).mean(), "= fraction of 1-2 character answers (should be near 0)")

# OPTIONAL rough chest filter by keywords (imperfect: spot-check by eye!)
KEYWORDS = ["chest", "lung", "thora", "pulmon", "pleur", "mediastin"]
mask = df[Q_COL].astype(str).str.lower().str.contains("|".join(KEYWORDS))
df = df[mask].reset_index(drop=True)
print(len(df), "rows after chest keyword filter")
df[[Q_COL, A_COL]].head()

train_2.csv: reconstructing file:   0%|          |  0.00B / 56.7MB            

train_2.csv: downloading bytes:           |  0.00B            

['index', 'Figure_path', 'Caption', 'Question', 'Choice A', 'Choice B', 'Choice C', 'Choice D', 'Answer', 'split'] 152603
0.009947379802494053 = fraction of 1-2 character answers (should be near 0)
3377 rows after chest keyword filter


,Question,ans
0,Where is the defect located in the lungs?,Right upper lobe
1,What did the chest CT images during hospitali...,High-density infiltrate in bilateral lungs and...
2,What did the chest CT images show after treat...,Absorption of the pleural effusion and inflamm...
3,What was the initial condition shown in the ch...,Low-density infiltrate in bilateral lungs and ...
4,What did the chest CT show after treatment?,Absorption of pleural effusion and inflammator...


## 3. Prompt

In [ ]:
import random, json, re

TYPES = {
    "wrong_organ": "replace the organ or anatomical structure with a different, incorrect one",
    "wrong_side": "flip the laterality (left/right) or location (upper/lower, etc.)",
    "invented_finding": "add a medical finding that is not supported by the correct answer",
    "wrong_attribute": "change a size, severity, count, or descriptive attribute to an incorrect one",
}

SYSTEM = ("You help build a research dataset for detecting hallucinations in medical image "
          "question answering. You output only valid JSON.")

TEMPLATE = '''Question: {q}
Correct answer: {a}

Task:
1. Write "correct_sentence": one natural, complete sentence that answers the question with the correct answer.
2. Write "hallucinated_sentence": the same sentence, but {instr}. Change as few words as possible and keep it fluent.
3. Write "hallucinated_spans": a list of the exact words from hallucinated_sentence that you changed or added (copy them exactly).

Return only JSON with keys correct_sentence, hallucinated_sentence, hallucinated_spans.'''

def ask_qwen(q, a, htype):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": TEMPLATE.format(q=q, a=a, instr=TYPES[htype])}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def validate(o):
    try:
        cs, hs, sp = o["correct_sentence"], o["hallucinated_sentence"], o["hallucinated_spans"]
        return bool(sp) and hs.strip() != cs.strip() and all(s in hs for s in sp)
    except Exception:
        return False

def word_labels(sentence, spans):
    ranges = []
    for s in spans:
        i = sentence.find(s)
        if i >= 0:
            ranges.append((i, i + len(s)))
    toks, labels = [], []
    for m in re.finditer(r"\S+", sentence):
        a, b = m.span()
        toks.append(m.group())
        labels.append(int(any(a < e and b > st for st, e in ranges)))
    return toks, labels

## 4. Quick test on a few rows (read the outputs yourself!)

In [ ]:
random.seed(0)
for _, r in df.head(5).iterrows():
    ht = random.choice(list(TYPES))
    raw = ask_qwen(r[Q_COL], r[A_COL], ht)
    o = parse_json(raw)
    print("TYPE:", ht, "| valid:", bool(o) and validate(o))
    print("Q:", r[Q_COL], "| A:", r[A_COL])
    print(raw, "\n")

TYPE: wrong_attribute | valid: True
Q:  Where is the defect located in the lungs?  | A: Right upper lobe
{
  "correct_sentence": "The defect is located in the right lower lobe of the lungs.",
  "hallucinated_sentence": "The defect is located in the left lower lobe of the lungs.",
  "hallucinated_spans": ["left", "lower"]
} 

TYPE: wrong_attribute | valid: False
Q:  What did the chest CT images during hospitalization show?  | A: High-density infiltrate in bilateral lungs and pleural effusion on the right side.
{
  "correct_sentence": "The chest CT images during hospitalization showed a high-density infiltrate in the left lung and pleural effusion on the left side.",
  "hallucinated_sentence": "The chest CT images during hospitalization showed a high-density infiltrate in the left lung and pleural effusion on the right side.",
  "hallucinated_spans": [
    "right side",
    "left side"
  ]
} 

TYPE: wrong_organ | valid: False
Q:  What did the chest CT images show after treatment?  | A: A

## 5. Full run
Writes both a hallucinated sample (some labels = 1) and a clean sample (all labels = 0) per row,
because your detector needs clean examples too. Rows where Qwen's output fails validation are skipped.

In [ ]:
from tqdm.auto import tqdm
import os
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/halluc_injected.jsonl"
N = 100   # test small first; None = all rows
sub = df if N is None else df.head(N)

random.seed(0)
kept = skipped = 0
with open(OUT, "w") as f:
    for idx, r in tqdm(sub.iterrows(), total=len(sub)):
        ht = random.choice(list(TYPES))
        o = parse_json(ask_qwen(r[Q_COL], r[A_COL], ht))
        if not o or not validate(o):
            skipped += 1
            continue
        t_h, l_h = word_labels(o["hallucinated_sentence"], o["hallucinated_spans"])
        t_c, l_c = word_labels(o["correct_sentence"], [])
        base = {"row": int(idx), "question": r[Q_COL], "gt_answer": str(r[A_COL]), "type": ht}
        f.write(json.dumps({**base, "answer": o["hallucinated_sentence"], "tokens": t_h, "labels": l_h, "hallucinated": 1}) + "\n")
        f.write(json.dumps({**base, "answer": o["correct_sentence"], "tokens": t_c, "labels": l_c, "hallucinated": 0}) + "\n")
        kept += 1
print("kept", kept, "skipped", skipped)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


  0%|          | 0/100 [00:00<?, ?it/s]

kept 77 skipped 23


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, pandas as pd
path = "/content/drive/MyDrive/halluc_injected.jsonl"
print(os.path.exists(path))

d = pd.read_json(path, lines=True)
print(len(d), "lines")
print(d.hallucinated.value_counts())

Mounted at /content/drive
True
154 lines
hallucinated
1    77
0    77
Name: count, dtype: int64


In [2]:
import pandas as pd
d = pd.read_json("/content/drive/MyDrive/halluc_injected.jsonl", lines=True)
h = d[d.hallucinated == 1]
print(d.hallucinated.value_counts(), d.type.value_counts(), sep="\n")
print("avg words per answer:", d.tokens.str.len().mean())
print("hallucinated share of words:", h.labels.apply(sum).sum() / h.tokens.str.len().sum())
print("has image column:", "image" in d.columns)

for _, r in h.sample(30, random_state=0).iterrows():
    print(r.gt_answer, "|", r.answer, "|", [t for t, l in zip(r.tokens, r.labels) if l])

hallucinated
1    77
0    77
Name: count, dtype: int64
type
invented_finding    56
wrong_side          36
wrong_attribute     32
wrong_organ         30
Name: count, dtype: int64
avg words per answer: 13.545454545454545
hallucinated share of words: 0.17177914110429449
has image column: False
Patchy opacifications with fibrotic shadows | The chest X-ray showed patchy opacifications with fibrotic shadows in the lower left lung. | ['lower', 'left']
4 years, enlarged cardiothoracic ratio, and left pulmonary artery | The previous chest radiography was performed 4 years ago and showed an enlarged cardiothoracic ratio and left pulmonary artery, and a new kidney tumor. | ['new', 'kidney', 'tumor.']
Day 15 and Day 22 | The images being compared in this caption are from Day 15, which shows significant improvement over Day 22. | ['Day', '15,', 'which', 'shows', 'significant', 'improvement']
An inflated intraoperative visual field | The image shows an enlarged intraoperative visual field in the upp